# Explorer 02 - Genres and styles

**Purpose.** Explore how genre and style composition moved from 2016 to 2026 and choose the fragmentation measure.

**Feeds main report section:** 5.1 RQ1 - Genre and style shifts

**Modules used:** `vinyl.trends`, `vinyl.viz`

**Tables read:** `release_genres`, `release_styles`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-02-styles")
from vinyl import trends, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_genres = ingest.read_table(spark, snapshot, "release_genres")
release_styles = ingest.read_table(spark, snapshot, "release_styles")

## 1. Coverage: how many releases have a genre or style at all?

In [ ]:
releases = ingest.read_table(spark, snapshot, "releases")
total = releases.groupBy("year").agg(F.count("*").alias("n_all"))
tagged = release_styles.groupBy("year").agg(F.countDistinct("release_id").alias("n_with_style"))
total.join(tagged, "year", "left").withColumn("coverage", F.col("n_with_style") / F.col("n_all")).orderBy("year").show()

## 2. Genre and style shares by year

In [ ]:
genre_shares = trends.tag_shares_by_year(release_genres, tag_col="genre").toPandas()
viz.plot_lines(genre_shares, "year", "share", "genre", title="TODO: genre shares", percent=True)

In [ ]:
style_shares = trends.tag_shares_by_year(release_styles, tag_col="style")
movers = trends.top_movers(style_shares, start_year=2016, end_year=2024, min_releases=500).toPandas()
movers.head(15), movers.tail(15)

## 3. Fragmentation: Shannon entropy

In [ ]:
entropy = trends.shannon_entropy_by_year(release_styles).toPandas()
entropy

In [ ]:
viz.plot_lines(entropy, "year", "evenness", title="TODO: style evenness by year")

## 4. Style co-occurrence networks

In [ ]:
cooc = trends.style_cooccurrence(release_styles, min_count=50)
cooc.where(F.col("year") == 2024).orderBy(F.desc("jaccard")).show(20, truncate=False)

In [ ]:
# TODO: compare network density / number of components / modularity per year.
# TODO: is rising entropy driven by new styles appearing, or by evener use of old ones?
# TODO: sensitivity of entropy to the undercounted years 2025-2026.

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] Use entropy or evenness as the headline fragmentation number, and why.
- [ ] End year for trend comparisons (2024, or later if lag is small).
- [ ] The five to eight styles worth plotting in the report.
- [ ] `min_count` threshold for co-occurrence edges.